In [1]:
import numpy as np
import igl
import pyvista as pv
pv.set_jupyter_backend('trame')

In [2]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [3]:
# Utility function to generate a tet grid
# n is a 3-tuple with the number of cell in every direction
# mmin/mmax are the grid bounding box corners

def tet_grid(n, mmin, mmax):
    nx = n[0]
    ny = n[1]
    nz = n[2]

    delta = mmax-mmin

    deltax = delta[0]/(nx-1)
    deltay = delta[1]/(ny-1)
    deltaz = delta[2]/(nz-1)

    T = np.zeros(((nx-1)*(ny-1)*(nz-1)*6, 4), dtype=np.int64)
    V = np.zeros((nx*ny*nz, 3))

    mapping = -np.ones((nx, ny, nz), dtype=np.int64)


    index = 0
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                mapping[i, j, k] = index
                V[index, :] = [i*deltax, j*deltay, k*deltaz]
                index += 1
    assert(index == V.shape[0])

    tets = np.array([
        [0,1,3,4],
        [5,2,6,7],
        [4,1,5,3],
        [4,3,7,5],
        [3,1,5,2],
        [2,3,7,5]
    ])

    index = 0
    for i in range(nx-1):
        for j in range(ny-1):
            for k in range(nz-1):
                indices = [
                    (i,   j,   k),
                    (i+1, j,   k),
                    (i+1, j+1, k),
                    (i,   j+1, k),

                    (i,   j,   k+1),
                    (i+1, j,   k+1),
                    (i+1, j+1, k+1),
                    (i,   j+1, k+1),
                ]

                for t in range(tets.shape[0]):
                    tmp = [mapping[indices[ii]] for ii in tets[t, :]]
                    T[index, :]=tmp
                    index += 1

    assert(index == T.shape[0])

    V += mmin
    return V, T

# Reading point cloud

In [4]:
# ----- Run for cat -----
pi, v = igl.read_triangle_mesh("data/cat.off")
pi /= 10
ni = igl.per_vertex_normals(pi, v)
to_pyvista_mesh(pi).plot()

Widget(value='<iframe src="http://localhost:55619/index.html?ui=P_0x1dc1a4b26d0_0&reconnect=auto" class="pyvis…

In [5]:
# ----- Run for luigi ------
pi, v = igl.read_triangle_mesh("data/luigi.off")
pi /= 10

mu = pi.mean(axis=0)
X = pi - mu

C = (X.T @ X) / X.shape[0]
evals, evecs = np.linalg.eigh(C) # Luigi's normals weren't rotating correctly, so I switched to eigh which gives sorted eigenvalues and orthogonal eigenvectors.
order = np.argsort(evals)[::-1]
R = evecs[:, order]          

# avoid reflection (mirroring flips orientation/normals)
if np.linalg.det(R) < 0:
    R[:, 2] *= -1

pi = (pi - mu) @ R + mu

ni = igl.per_vertex_normals(pi, v)

to_pyvista_mesh(pi).plot()

Widget(value='<iframe src="http://localhost:55619/index.html?ui=P_0x1dc1a881550_1&reconnect=auto" class="pyvis…

In [6]:
def find_closest_point(point, points):
    d2 = np.sum((points - point[None, :])**2, axis=1)
    return int(np.argmin(d2))

bbox_min = pi.min(axis=0)
bbox_max = pi.max(axis=0)
bbox_diag = np.linalg.norm(bbox_max - bbox_min)
eps0 = 0.01 * bbox_diag 

# Normalize normals
n = ni / (np.linalg.norm(ni, axis=1, keepdims=True) + 1e-12)

P_on = pi.copy()
P_out = np.zeros_like(pi)
P_in  = np.zeros_like(pi)

for i in range(pi.shape[0]):
    p = pi[i]
    nn = n[i]

    # outside: p + eps * n
    eps = eps0
    while True:
        q = p + eps * nn
        if find_closest_point(q, pi) == i:
            break
        eps *= 0.5
        if eps < 1e-8:
            break
    P_out[i] = q

    # inside: p - eps * n
    eps = eps0
    while True:
        q = p - eps * nn
        if find_closest_point(q, pi) == i:
            break
        eps *= 0.5
        if eps < 1e-8:
            break
    P_in[i] = q

# Stack all constraint locations: (3n, 3)
P = np.vstack([P_on, P_out, P_in])

d_out = np.linalg.norm(P_out - P_on, axis=1)   
d_in  = np.linalg.norm(P_in  - P_on, axis=1)   

f = np.concatenate([
    np.zeros(P_on.shape[0], dtype=float),  # f(pi) = 0
    +d_out.astype(float),                  # f(pi+) > 0
    -d_in.astype(float),                   # f(pi-) < 0
])  

# Build per-point RGB colors for P = [P_on; P_out; P_in]
n0 = P_on.shape[0]
colors = np.vstack([
    np.tile([0, 0, 255], (n0, 1)),  # surface: blue
    np.tile([255, 0, 0], (n0, 1)),  # outside: red
    np.tile([0, 255, 0], (n0, 1)),  # inside: green
]).astype(np.uint8)

# Display constraints point cloud
cloud = pv.PolyData(P)
cloud["rgb"] = colors
cloud.plot(
    scalars="rgb",
    rgb=True,
    render_points_as_spheres=True,
    point_size=8
)

Widget(value='<iframe src="http://localhost:55619/index.html?ui=P_0x1dc1a881a90_2&reconnect=auto" class="pyvis…

# MLS function

In [7]:
resolution = 30

# Axis-aligned bounding box of the point cloud
bbox_min = pi.min(axis=0)
bbox_max = pi.max(axis=0)

pad = 0.10 * (bbox_max - bbox_min)
mmin = bbox_min - pad
mmax = bbox_max + pad

# Build regular volumetric tet grid
x, T = tet_grid((resolution, resolution, resolution), mmin, mmax)

In [8]:
def closest_points(point, points, h):
    # indices of points within distance < h
    d2 = np.sum((points - point[None, :])**2, axis=1)
    return np.argwhere(d2 < h*h).ravel()

def wendland_weight(r, h):
    # w(r) = (1 - r/h)^4 * (4 r/h + 1) for r < h, else 0
    w = np.zeros_like(r, dtype=float)
    mask = r < h
    t = 1.0 - (r[mask] / h)
    w[mask] = (t**4) * (4.0 * (r[mask] / h) + 1.0)
    return w

def poly_basis(P, degree):
    # P: (n,3) -> A: (n,m)
    x = P[:, 0]
    y = P[:, 1]
    z = P[:, 2]
    n = P.shape[0]

    if degree == 0:
        return np.ones((n, 1), dtype=float)

    if degree == 1:
        # [1, x, y, z]
        return np.column_stack([np.ones(n), x, y, z]).astype(float)

    if degree == 2:
        # [1, x, y, z, x^2, y^2, z^2, xy, xz, yz]
        return np.column_stack([
            np.ones(n), x, y, z,
            x*x, y*y, z*z,
            x*y, x*z, y*z
        ]).astype(float)


def mls_eval(xi, P, f, wendlandRadius, polyDegree, outside_value=1e3, reg=1e-12):
    """
    Evaluate MLS implicit function at xi by solving:
      minimize_c sum_j w(||Pj - xi||) (phi(Pj)^T c - f_j)^2
    and returning phi(xi)^T c
    """
    # number of polynomial coefficients
    m = poly_basis(np.zeros((1, 3)), polyDegree).shape[1]

    idx = closest_points(xi, P, wendlandRadius)
    if idx.size < 2 * m:
        return float(outside_value)

    Pn = P[idx]
    fn = f[idx]

    r = np.linalg.norm(Pn - xi[None, :], axis=1)
    w = wendland_weight(r, wendlandRadius)

    keep = w > 0
    if np.count_nonzero(keep) < 2 * m:
        return float(outside_value)

    Pn = Pn[keep]
    fn = fn[keep]
    w = w[keep]

    A = poly_basis(Pn, polyDegree)          
    sw = np.sqrt(w)                         
    Aw = A * sw[:, None]                     
    bw = fn * sw                           

    # Solve weighted least squares. @ = matrix multiplication
    M = Aw.T @ Aw
    rhs = Aw.T @ bw
    M.flat[::M.shape[0] + 1] += reg

    try:
        c = np.linalg.solve(M, rhs)
    except np.linalg.LinAlgError:
        c = np.linalg.lstsq(Aw, bw, rcond=None)[0]

    return float((poly_basis(xi[None, :], polyDegree) @ c)[0])

In [9]:
polyDegree = 2 # 0, 1, or 2 will break if not 0-2

wendlandRadius = 0.22 * bbox_diag  # 0.10–0.25 * bbox_diag if needed

fx = np.zeros(x.shape[0], dtype=float)
for i in range(x.shape[0]):
    fx[i] = mls_eval(
        x[i],
        P, f,
        wendlandRadius=wendlandRadius,
        polyDegree=polyDegree,
        outside_value=1e3
    )

ind = np.where(fx < 0, -1, 1)
to_pyvista_mesh(x).plot(scalars=ind)

Widget(value='<iframe src="http://localhost:55619/index.html?ui=P_0x1dc1a86da50_3&reconnect=auto" class="pyvis…

# Marching to extract surface

In [11]:
grid = pv.UnstructuredGrid({pv.CellType.TETRA: T}, x)
grid["fx"] = fx

surface = grid.contour([0.0], scalars="fx").triangulate()

sv = surface.points
sf = surface.faces.reshape(-1, 4)[:, 1:]

to_pyvista_mesh(sv, sf).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:55619/index.html?ui=P_0x1dc28545810_4&reconnect=auto" class="pyvis…

In [ ]:
grid = pv.UnstructuredGrid({pv.CellType.TETRA: T}, x)
grid["fx"] = fx

surface = grid.contour([0.0], scalars="fx").triangulate()

sv = surface.points
sf = surface.faces.reshape(-1, 4)[:, 1:]

n_components, comp = igl.facet_components(sf)

counts = np.bincount(comp)
largest = int(np.argmax(counts))

# Keep only faces in largest component
sf_keep = sf[comp == largest]

# Trim vertices
used, inv = np.unique(sf_keep.ravel(), return_inverse=True)
sv_keep = sv[used]
sf_keep = inv.reshape(-1, 3)

to_pyvista_mesh(sv_keep, sf_keep).plot(show_edges=True)

ValueError: setting an array element with a sequence. The requested array would exceed the maximum number of dimension of 1.